# Preprocessing — Member 4 (second turn): Class Imbalance Experiment & PageValues Decision
## Online Shopper Purchase Likelihood Prediction System
**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

This is the last step of the preprocessing relay. It builds on:
- **Member 4 (turn 1):** validated, de-duplicated data and the stratified 80/20 split
- **Member 1:** engineered features and log-transformed skewed columns (`data/processed/X_train_engineered.csv`)
- **Member 2:** `build_preprocessor()` (one-hot encoding + scaling) from `src/preprocessing/encoding_scaling.py`
- **Member 3:** feature selection result (keep all 13 numeric features)

Two questions are answered here, both with cross-validation on the **training set only**:
1. Which class-imbalance strategy works best: no handling, class weighting, or SMOTE?
2. What happens to model performance with and without `PageValues`, and what should we do about it?

**The test set is not loaded anywhere in this notebook.** It stays untouched for the final evaluation in Stage 6/7.

**Prediction point:** the prediction is made at any point during an active session, before checkout is completed.


## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# imbalanced-learn provides SMOTE and a Pipeline that only resamples inside training folds.
# If this import fails, run:  pip install imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from src.preprocessing.encoding_scaling import CATEGORICAL_COLUMNS, build_preprocessor

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## Load the training data
Only training data is loaded. Member 1's engineered file is used (before encoding and scaling), because the
encoder and scaler are fitted **inside each cross-validation fold** below. Loading the already-scaled file
instead would let each validation fold influence the scaler's mean and standard deviation, which is a small
but real leak.

In [ ]:
X_train = pd.read_csv("../data/processed/X_train_engineered.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)

assert len(X_train) == len(y_train)

numeric_columns = [c for c in X_train.columns if c not in CATEGORICAL_COLUMNS]

print("Training rows:", len(X_train))
print("Purchase rate in training data: {:.3f}".format(y_train.mean()))
print("Numeric features:", len(numeric_columns), "| Categorical features:", len(CATEGORICAL_COLUMNS))


## Experiment setup
- **Validation:** 5-fold stratified cross-validation, so every fold keeps the ~15.5% purchase rate.
- **Models:** Logistic Regression and Random Forest. The best imbalance method can differ by algorithm, so
  testing two very different model types is more informative than testing one.
- **Metrics (purchase class):** F1, precision, recall, ROC-AUC and PR-AUC. Accuracy is deliberately left out:
  a model that always predicts "no purchase" already scores about 84.5%.
- **No leakage:** the preprocessor and SMOTE are steps inside an `imblearn` Pipeline, so they are fitted on each
  fold's training portion only. SMOTE never touches a validation fold.
- **F1, precision and recall use the default 0.5 decision threshold.** ROC-AUC and PR-AUC do not depend on a threshold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


def make_model(model_name, weighted):
    class_weight = "balanced" if weighted else None
    if model_name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=RANDOM_STATE)
    return RandomForestClassifier(n_estimators=200, class_weight=class_weight,
                                  random_state=RANDOM_STATE, n_jobs=-1)


def make_pipeline(method, model_name, numeric_cols):
    steps = [("prep", build_preprocessor(numeric_cols))]
    if method == "SMOTE":
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("model", make_model(model_name, weighted=(method == "Class weighting"))))
    return ImbPipeline(steps)


def evaluate(method, model_name, X, numeric_cols):
    scores = cross_validate(make_pipeline(method, model_name, numeric_cols), X, y_train,
                            cv=cv, scoring=SCORING)
    row = {}
    for metric in SCORING:
        values = scores["test_" + metric]
        row[metric] = values.mean()
        row[metric + "_std"] = values.std()
    return row


## Experiment 1 — Class imbalance strategy

In [ ]:
rows = []
for model_name in ["Logistic Regression", "Random Forest"]:
    for method in ["Baseline", "Class weighting", "SMOTE"]:
        result = evaluate(method, model_name, X_train, numeric_columns)
        rows.append({"model": model_name, "method": method, **result})

imbalance_results = pd.DataFrame(rows)
imbalance_results[["model", "method", "f1", "precision", "recall", "roc_auc", "pr_auc"]].round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model_name in zip(axes, ["Logistic Regression", "Random Forest"]):
    subset = imbalance_results[imbalance_results["model"] == model_name]
    ax.bar(subset["method"], subset["f1"], yerr=subset["f1_std"], color=["#4C72B0", "#DD8452", "#1F7A6C"], capsize=4)
    ax.set_title(model_name)
    ax.set_ylabel("F1 (purchase class), 5-fold CV")
    ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()


In [ ]:
imbalance_results[["model", "method", "f1", "f1_std", "precision", "precision_std", "recall", "recall_std"]].round(3)

### Interpretation — imbalance strategy

*(The numbers below are approximate. Check them against the tables above from your own run and correct any that differ.)*

**Logistic Regression:** class weighting (F1 about 0.65) and SMOTE (about 0.64) score somewhat higher than the
baseline (about 0.62), though the F1 gap is modest compared with the fold-to-fold spread. The clearer effect is on recall, which rises from about 0.55 to about 0.81, while precision falls
from about 0.71 to about 0.54. Class weighting and SMOTE are very close to each other; a gap of about 0.005
is smaller than the fold-to-fold spread (the `f1_std` column), so it should not be read as one method being
truly better.

**Random Forest:** the picture is different. SMOTE is clearly best (F1 about 0.67 versus about 0.62 baseline),
and it balances precision (about 0.67) and recall (about 0.68). Class weighting does **not** help here (F1
about 0.60, slightly below baseline). A likely reason is that forest trees grow until their leaves are almost
pure, so reweighting the classes changes very little; this explanation is an inference and was not tested here.

**Two things worth noting:**
- **ROC-AUC barely moves** (about 0.91 to 0.92 for every setting), and **PR-AUC is highest for the baseline**
  in both models. This suggests that weighting and SMOTE mostly shift the model's decision threshold (trading
  precision for recall) rather than improving its ability to rank purchasers above non-purchasers.
- **So the choice depends on the business goal.** If missing a likely purchaser is costly (for example,
  missing a chance to offer help), the higher-recall settings are preferable. If contacting non-buyers is
  costly, the higher-precision baseline may be better. This should be decided with the stakeholders' needs in
  mind, not by F1 alone.

**Limitation:** SMOTE is applied after one-hot encoding, so synthetic rows can contain fractional values in
one-hot columns (for example 0.4 for a category). `SMOTENC`, which handles categorical columns properly, is a
candidate to compare in Stage 6.

**Decision:** do not lock in one method now. Carry **baseline, class weighting and SMOTE** into Stage 6 as
settings to compare per algorithm, together with threshold tuning, because the best choice differs by model.

## Experiment 2 — `PageValues`: with and without

`PageValues` was flagged in EDA as the strongest predictor and as a leakage risk. Here the same models are
compared with and without it, using class weighting (kept fixed so that only the feature set changes).
When `PageValues` is removed it is also removed from the scaler's numeric column list.

In [ ]:
numeric_no_pv = [c for c in numeric_columns if c != "PageValues"]
X_no_pv = X_train.drop(columns=["PageValues"])

rows = []
for model_name in ["Logistic Regression", "Random Forest"]:
    with_pv = evaluate("Class weighting", model_name, X_train, numeric_columns)
    without_pv = evaluate("Class weighting", model_name, X_no_pv, numeric_no_pv)
    rows.append({"model": model_name, "features": "With PageValues", **with_pv})
    rows.append({"model": model_name, "features": "Without PageValues", **without_pv})

pagevalues_results = pd.DataFrame(rows)
pagevalues_results[["model", "features", "f1", "precision", "recall", "roc_auc", "pr_auc"]].round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ["f1", "roc_auc"], ["F1 (purchase class)", "ROC-AUC"]):
    sns.barplot(data=pagevalues_results, x="model", y=metric, hue="features",
                palette=["#1F7A6C", "#DD8452"], ax=ax)
    ax.set_title(title)
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


### Interpretation — `PageValues`

*(Approximate numbers; check them against the table above from your own run.)*

Removing `PageValues` causes a very large drop in performance:
- **Logistic Regression:** F1 falls from about 0.65 to about 0.40; ROC-AUC from about 0.92 to about 0.75.
- **Random Forest:** F1 falls from about 0.60 to about 0.13 (recall about 0.07, so it finds almost no
  purchasers at the default threshold); ROC-AUC from about 0.92 to about 0.77.

So `PageValues` carries most of the predictive signal in this dataset. That fact alone does not tell us whether
it is *legitimate* signal. A feature this dominant is exactly what we would expect to see if it contains
information about the outcome itself. The dataset comes from Google Analytics, whose "page value" metric is
calculated from the transaction revenue attributed to pages; if the values in this dataset were computed that
way, they may partly reflect whether the session ended in a purchase. This needs to be checked against the
dataset documentation and cannot be settled from the data alone.

There is also a practical question for the deployed system: a user of the prediction app would have to
supply a `PageValues` figure. If that number cannot be measured in real time before checkout, the feature
cannot be used in production, whatever its accuracy benefit.

### Decision

**The availability of `PageValues` at the intended prediction time is uncertain.** Therefore, models will be
compared with and without `PageValues`. It will only be included in the deployed model if it is confirmed to
be available before prediction and does not introduce data leakage.

What this means in practice for Stage 6/7:
- Every candidate model is trained and reported on **both** feature sets.
- The "without PageValues" results are the honest estimate of what the system can do if `PageValues` is
  ruled out, and should be reported next to the "with" results, not hidden.
- Before the final model is chosen, the group must confirm (from the UCI documentation and the original
  paper) how `PageValues` was computed, and record that answer in the report.

---
## Summary of this step

| Question | Result | Carried into Stage 6 |
|---|---|---|
| Best imbalance strategy | Depends on the model: class weighting ≈ SMOTE for Logistic Regression; SMOTE best for Random Forest; weighting does not help Random Forest | Compare baseline, class weighting and SMOTE per algorithm, plus threshold tuning |
| Effect of removing `PageValues` | Very large drop for both models | Report every model with and without it |
| Is `PageValues` safe to use? | Uncertain — not settled by the data | Include in the deployed model only if confirmed available before prediction and leakage-free |

**Preprocessing pipeline is now complete:** validation and de-duplication (Member 4) → split (Member 4) →
feature engineering and log transform (Member 1) → encoding and scaling (Member 2) → feature selection
(Member 3) → imbalance and `PageValues` experiments (Member 4).
